In [44]:
import os
from pyoxigraph import Store, RdfFormat
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from kneed import KneeLocator

from elasticsearch import Elasticsearch

from utils import ollama_request, EMBEDD_MODEL_1

import spacy
nlp = spacy.load("en_core_web_sm")

INDEX_NAME = os.getenv("INDEX_NAME")
ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
FRAGMENT_INDEX_NAME = f"{INDEX_NAME}_fragments"
TRIPLETS_INDEX_NAME = f"{INDEX_NAME}_triplets_index"

EMBEDDING_MODEL = EMBEDD_MODEL_1

es_client = Elasticsearch('http://localhost:9200')

graph = Store()
graph.load(
    path=f"{INDEX_NAME}.ttl",
    format=RdfFormat.TURTLE
)

In [47]:
def extract_objects(text):
    doc = nlp(text)

    subjects = []
    target_deps = {
        "nsubj",
        "nsubjpass",
        "dobj",
        "pobj",
        "iobj",
    }
    for token in doc:
        if token.dep_ in target_deps:
            subject = " ".join(
                t.text for t in token.subtree
            )
            subjects.append(subject)

    return subjects

In [48]:
def execute_query(graph, start_date, end_date, filtering_criteria):
    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id ?triplet_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:triplet_id ?triplet_id ;
          ns1:speech_id ?speech_id .

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
        {filtering_criteria}
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        filtering_criteria=filtering_criteria
    )

    return graph.query(query)

In [49]:
def get_df_from_query_result(query_res):
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
    }
    for row in query_res:
        fragment_start = row.start
        fragment_end = row.end
        speech_id = row.speech_id

        triplets_resp["subject"].append(row.s.split('/')[-1])
        triplets_resp["predicate"].append(row.p.split('/')[-1])
        triplets_resp["object"].append(row.o.split('/')[-1])
        triplets_resp["date"].append(str(row.date.value))
        triplets_resp["start"].append(fragment_start)
        triplets_resp["end"].append(fragment_end)
        triplets_resp["speech_id"].append(speech_id)
    return pd.DataFrame(triplets_resp)

In [50]:
def get_triplets_by_id(
    graph,
    triplet_ids,
    start_date="1900-01-01",
    end_date="2100-01-01"
):
    triplet_ids_values = " ".join(str(x) for x in triplet_ids)

    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:speech_id ?speech_id ;
          ns1:triplet_id ?triplet_id .

    VALUES ?triplet_id {{ {triplet_ids_values} }}

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        triplet_ids_values=triplet_ids_values,
    )

    return graph.query(query)

In [51]:
def knee_cutoff(scores):
    scores = np.sort(np.asarray(scores))[::-1]

    indices = np.arange(len(scores))

    knee = KneeLocator(
        indices,
        scores,
        curve="convex",
        direction="decreasing"
    )

    return int(knee.knee) - 1

In [52]:
def score_df_cutoff(df, top_k=10):
    df = df.iloc[:top_k, :].sort_values(by="score", ascending=False)

    sorted_scores = df["score"].values
    if len(sorted_scores) < 2:
        return df
    gaps = sorted_scores[:-1] - sorted_scores[1:]
    cutoff_1 = np.argmax(gaps) + 1
    cutoff_2 = knee_cutoff(sorted_scores)

    if cutoff_2 == 0:
        cutoff = cutoff_1
    else:
        cutoff = cutoff_2
    return df.iloc[:cutoff, :]

In [53]:
def find_similar_entities(entity, score_threshold=0.5, top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(entity)
    final_res = {
        "score": [],
        "value_name": [],
    }
    response = es_client.search(
        index=ENT_INDEX_NAME,
        size=100,
        knn={
            "field": "value_embedding",
            "query_vector": query_embedding.tolist(),
            "k": 100,
            "num_candidates": 100,
            "filter": {
                "match_all": {}
            }
        },
        source=["value_name"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["value_name"].append(result["_source"]["value_name"])
    
    final_res = pd.DataFrame(final_res)
    return score_df_cutoff(final_res, top_k=top_k)


In [54]:
def find_question_related_triplets(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01", top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(question)
    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": query_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "match_all": {}
            }
        },
        source=["triplet_id"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    related_triplets = pd.DataFrame(final_res)
    related_triplets = score_df_cutoff(related_triplets, top_k=top_k)
    related_ids = list(related_triplets["triplet_id"])

    query_res = get_triplets_by_id(graph, related_ids, start_date=start_date, end_date=end_date)
    res_df = get_df_from_query_result(query_res)
    res_df["information_type"] = "main"

    return res_df


In [ ]:
def find_additional_related_triplets(graph, question, score_threshold=0.5, top_entity_k=10, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    question_objects = extract_objects(question)
    question_embedding = EMBEDDING_MODEL.encode(question)
    similar_entities = []
    for object in question_objects:
        similar_entities = similar_entities + list(find_similar_entities(object, score_threshold=score_threshold, top_k=top_entity_k)["value_name"].values)

    filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
        "triplet_id": [],
    } 
    for ent in similar_entities:
        try:
            query_res = list(
                execute_query(
                    graph=graph,
                    start_date=start_date,
                    end_date=end_date,
                    filtering_criteria=filtering_criteria.format(val=ent),
                )
            )
            print(f"Found {len(query_res)} triplets for entity: {ent}")
        except Exception as e:
            continue

        for row in query_res:
            fragment_start = row["start"]
            fragment_end = row["end"]
            speech_id = row["speech_id"]

            triplet_id = row["triplet_id"]
            
            triplets_resp["subject"].append(row.s.split('/')[-1])
            triplets_resp["predicate"].append(row.p.split('/')[-1])
            triplets_resp["object"].append(row.o.split('/')[-1])
            triplets_resp["date"].append(str(row.date.value))
            triplets_resp["start"].append(fragment_start)
            triplets_resp["end"].append(fragment_end)
            triplets_resp["speech_id"].append(speech_id)
            triplets_resp["triplet_id"].append(triplet_id)

    triplets_df = pd.DataFrame(triplets_resp)

    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": question_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "terms": {
                    "triplet_id": triplets_df["triplet_id"].tolist()
                }
            }
        },
        source=["triplet_id"]
    )
    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    final_scores = pd.DataFrame(final_res)
    final_scores["triplet_id"] = final_scores["triplet_id"].astype("int64")
    triplets_df["triplet_id"] = triplets_df["triplet_id"].astype("int64")

    triplets_df = triplets_df.merge(
        final_scores,
        on="triplet_id",
        how="inner"
    )
    triplets_df = score_df_cutoff(triplets_df, top_k=top_triplet_k)

    triplets_df["information_type"] = "additional"
    triplets_df = triplets_df.drop(columns=["score", "triplet_id"])
    return triplets_df

In [56]:
def get_prompt(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01"):

    main_triplets_related_to_question = find_question_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_k=1000
    )

    additional_triplets_related_to_question = find_additional_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_entity_k=10,
        top_triplet_k=1000
    )

    final_triplets = pd.concat([main_triplets_related_to_question, additional_triplets_related_to_question], ignore_index=True)

    final_triplets = final_triplets.drop_duplicates(subset=["subject", "predicate", "object", "date", "start", "end", "speech_id"])
    final_triplets = final_triplets.sort_values(by=["date", "start"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    main_triplets = final_triplets[final_triplets["information_type"] == "main"]
    additional_triplets = final_triplets[final_triplets["information_type"] == "additional"]

    prompt = QUESTION_ANSWER_PROMPT.format(
        question=question,
        main_facts=main_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records"),
        additional_facts=additional_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")
    )

    return prompt, final_triplets

In [57]:
def get_fragment(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    start = int(row.start)
    end = int(row.end) + 1
    speech_id = row.speech_id
    return ".".join(es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"].split(".")[start:end])

def get_speech(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    speech_id = row.speech_id - 1

    return es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

In [58]:
QUESTION_ANSWER_PROMPT = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.
Each fact contains an index.
You can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".
Use just syntax with [index] to refer to the fact, do not use any additional words.

Facts are split into two categories: main and additional.
Main facts (most important) are directly related to the research question. 
Additional facts provide context or background information for entities present in the question.

Research question: {question}

Main Facts:
{main_facts}

Additional Facts:
{additional_facts}

Your task is to answer the research question based on the provided facts."
"""

In [59]:
QUESTION = """
When do numerous references to the Russian world (russkiy mir) begin to appear?
"""

START_DATE = "1999-01-01"
END_DATE = "2024-12-31"

PROMPT, final_triplets = get_prompt(
    graph=graph,
    question=QUESTION,
    score_threshold=0.5,
    start_date=START_DATE,
    end_date=END_DATE
)
print(f"Used main facts: {len(final_triplets[final_triplets['information_type'] == 'main'])}")
print(f"Used additional facts: {len(final_triplets[final_triplets['information_type'] == 'additional'])}")
PROMPT

Found 3 triplets for entity: russian_history


AttributeError: 'pyoxigraph.QuerySolution' object has no attribute 'start'

In [ ]:
res = ollama_request(
    prompt=PROMPT, is_stream=False
)
 
display(Markdown(res))

In [23]:
REFERENCE_NUM = 13

display(final_triplets[final_triplets["index"] == REFERENCE_NUM][["subject", "predicate", "object"]].reset_index(drop=True))

display(Markdown("### Reference Fragment"))
display(Markdown(get_fragment(final_triplets, REFERENCE_NUM)))

display(Markdown("### Reference Speech"))
display(Markdown(get_speech(final_triplets, REFERENCE_NUM)))


,subject,predicate,object
0,vladimir_putin,considered_offensive_and_defensive_systems_as_...,russia


### Reference Fragment

 We were talking about the possible kinds and versions of response in the event that one side comes out unilaterally. I was not talking about increasing the missiles. I was talking about how you would substitute single-unit warheads, make them MIRV warheads

### Reference Speech

My general impression is very favourable. I think this summit is even better than the previous, in which I also took part. True, we worked according to a plan—but I find it not so over-organised. There is not so much red tape now. So we often digressed to related matters. We had a very interesting and positive discussion, which concerned my country’s interests in many fields, which was of great importance to me. We discussed bilateral contacts with many countries, and international affairs. Everything we discussed was of tremendous interest to Russia. We had the opportunity not merely to speak up and learn our partners’ stances but also to co-ordinate team efforts for the development of international relations as a whole and of bilateral contacts and certain topical issues—some of them critical. I think it all deserves approval. Bilateral meetings are part of the routine of G8 summits as they provide a very convenient form of contact and of exploring bilateral problems. That is why we took advantage of this opportunity here. I met with the federal chancellor last night, and with the British and Italian prime ministers today. Though bilateral meetings usually concern bilateral relations, other issues also come under discussion when international coordination of efforts is of special interest. I discussed mainly economic matters with the German chancellor and the Italian prime minister. There was ample room for discussion here as Germany and Italy are Russia’s principal trading and economic partners, Germany leading for Europe, and Italy a close second. The pre-crisis level of Russian-Italian trade turnover has now been exceeded —over $9 billion last year, as against $6 billion in 1998. There are ambitious bilateral projects—not only in the long-established energy sphere but also in aircraft building and some other fields. That was why Mr Berlusconi and I discussed those topics. As for Mr Tony Blair, we also took up certain international issues, in particular, the Middle East, Iraq and the Balkans. All summit participants also discussed the Balkan problem at lunch today, to my great satisfaction. Though opinions clashed outside the field of our interaction, the lunch proceeded in an informal and businesslike atmosphere of openness. It was of great help in our work, so we agreed on many things, and coordinated many questions between ourselves. I discussed the entire range of Russian-Japanese issues, including frontier delineation, with Mr Koizumi. That was our first meeting—we had only had telephone conversations before. Of greatest importance, was that both parties agreed to stick to all previous agreements, including those which Mr Mori, his predecessor, and I made in Irkutsk. Future bilateral relations will proceed from that. I shall meet with the US President tomorrow, so I think it would be wiser to discuss that meeting tomorrow or after I come back to Moscow. The United States is Russia’s key partner, so we shall have many things to discuss, especially our economic links. As you know, the US Secretary of Commerce and Secretary of the Treasury are coming to Moscow quite soon according to our Ljubljana agreement. American businessmen will also visit Russia. We shall talk about the prospects of our next meeting in the United States, most probably, this autumn. We shall surely also discuss international stability and security—the entire range of the complicated issues, including those connected with the 1972 ABM Treaty. I certainly cannot inform you about the achievements of this meeting before it finishes. I liked today’s discussion of the Balkans and Macedonia. I liked our partners’ attitudes and their principled approach to the settlement of this extremely involved problem. However complicated the problems we encounter might be, there is one criterion on which all agree. That is the inviolability of frontiers in the region, and the preservation of sovereignty. Macedonia is no exception here. That is what matters most, and that is the only basis on which all other issues can be settled, including language, culture, and so on. I fully agree with this. We support such an approach, and we shall co-operate with our partners in finding a settlement in Macedonia and the entire Balkans.